In [1]:
# pull_openAP.py
# loads and cleans anomaly portfolio returns from Open Source Asset Pricing
# Chen & Zimmermann (2022), Critical Finance Review
#
# data file: PredictorLSretWide.csv  (212 long-short monthly returns)
# download from: https://www.openassetpricing.com/data/
# output: X_openAP.csv

# %%

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

START = "1963-07"
END   = "2023-12"


# %%

#___ load raw file ___

print("loading PredictorLSretWide.csv...")
df_raw = pd.read_csv("PredictorLSretWide.csv")
print(f"  shape: {df_raw.shape}")
print(f"  first few cols: {df_raw.columns[:5].tolist()}")
print(f"  sample dates: {df_raw.iloc[:3, 0].tolist()}")


# %%

#___ parse dates ___

"""
openAP uses month-end dates like '1926-01-30' -- converting to period
month so everything aligns properly when we subtract the risk-free rate
later (e.g. 1963-07-31 and 1963-07-01 both map to Period('1963-07'))
"""

date_col = df_raw.columns[0]
df_raw[date_col] = pd.to_datetime(df_raw[date_col])
df_raw.index = df_raw[date_col].dt.to_period("M")
df_raw = df_raw.drop(columns=[date_col])
df_raw.index.name = "date"

print(f"date range in file: {df_raw.index[0]} to {df_raw.index[-1]}")


# %%

#___ check scale ___

# some data sources ship returns as percentages, need to confirm
sample_vals = df_raw.iloc[400:450, :30].values.flatten()
sample_vals = sample_vals[~np.isnan(sample_vals.astype(float))]
avg_abs = np.abs(sample_vals.astype(float)).mean()
print(f"avg absolute value mid-sample: {avg_abs:.4f}")
if avg_abs > 1:
    print("  looks like percentages -- dividing by 100")
    df_raw = df_raw / 100
else:
    print("  already decimal, no conversion needed")


# %%

# filter to sample period and run a quick coverage analysis

df = df_raw.loc[START:END].copy()
print(f"after date filter: {df.shape}")

total_months = len(df)
coverage = df.notna().sum()

print(f"\ncoverage summary:")
print(f"  total months:          {total_months}")
print(f"  total anomalies:       {df.shape[1]}")
print(f"  full coverage (100%):  {(coverage == total_months).sum()}")
print(f"  90%+ coverage:         {(coverage >= 0.90 * total_months).sum()}")
print(f"  75%+ coverage:         {(coverage >= 0.75 * total_months).sum()}")

first_valid = df.apply(lambda c: c.first_valid_index())
yr_dist = first_valid.map(lambda p: p.year if p is not None else None).value_counts().sort_index()
print(f"\nanomalies by start year:")
for yr, cnt in yr_dist.items():
    bar = "█" * min(int(cnt), 50)
    print(f"  {yr}: {bar} ({cnt})")


# %%

#___ build X matrix -- keep anomalies with 90%+ coverage ___

good_cols = coverage[coverage >= 0.90 * total_months].index.tolist()
X_wide    = df[good_cols].copy()
print(f"anomalies passing 90% threshold: {len(good_cols)}")

# forward fill small gaps (up to 3 months), then drop anything still missing
X_wide = X_wide.fillna(method="ffill", limit=3)
X_wide = X_wide.dropna(how="any")
print(f"after cleaning: {X_wide.shape[0]} months x {X_wide.shape[1]} anomalies")
print(f"date range: {X_wide.index[0]} to {X_wide.index[-1]}")


# %%

#___ subtract risk-free rate ___

"""
using Fama-French RF from pandas_datareader -- need to make sure the
period index on RF matches X_wide exactly, otherwise the subtraction
silently produces all NaNs (took a while to debug this)
"""

try:
    import pandas_datareader.data as web
    print("fetching RF from Fama-French...")
    ff = web.DataReader("F-F_Research_Data_Factors", "famafrench",
                        start="1963-01-01", end="2024-01-01")[0]
    if not isinstance(ff.index, pd.PeriodIndex):
        ff.index = ff.index.to_period("M")
    rf = ff["RF"] / 100

    rf_aligned = rf.reindex(X_wide.index)
    missing_rf = rf_aligned.isna().sum()
    if missing_rf > 0:
        print(f"  {missing_rf} months missing RF -- filling with 0")
        rf_aligned = rf_aligned.fillna(0)

    X_excess = X_wide.subtract(rf_aligned, axis=0)
    print(f"  excess returns matrix: {X_excess.shape}")

    if len(X_excess) == 0:
        print("  alignment failed, falling back to raw returns")
        X_excess = X_wide.copy()

except Exception as e:
    print(f"  could not get RF ({e}), using raw returns")
    X_excess = X_wide.copy()


# %%

# sanity check -- top anomalies by annualised SR and overlap with paper

means  = X_excess.mean()
stds   = X_excess.std(ddof=1)
sr_ann = (means / stds) * np.sqrt(12)
sr_sorted = sr_ann.sort_values(ascending=False)

print(f"top 20 anomalies by annualised SR:")
print(f"  {'anomaly':35s}  {'mean ret %':>10s}  {'SR':>8s}")
print("  " + "-" * 57)
for name in sr_sorted.head(20).index:
    print(f"  {name:35s}  {means[name]*100:>10.3f}%  {sr_sorted[name]:>8.3f}")

print(f"\nSR stats across all anomalies:")
print(f"  mean:     {sr_ann.mean():.3f}")
print(f"  median:   {sr_ann.median():.3f}")
print(f"  % > 0:    {(sr_ann > 0).mean()*100:.1f}%")


# %%

# check how much we overlap with the 37 anomalies from the paper
# useful to understand why our replication SR gap is smaller than theirs

paper_37 = [
    "BM", "EP", "CFP", "DP", "Mom12m", "STreversal", "LTreversal",
    "Investment", "GP", "Size", "Mom6m", "Accruals", "NOA", "AssetGrowth",
    "InvestPPEInv", "Leverage", "IdioVol3F", "IndMom", "ValMomProf",
    "ValMom", "ValProf", "MomRev", "IndMomRev", "IndustryMom",
    "Season", "CISS", "DivP", "SP", "roaq", "roaeq"
]
overlap = [p for p in paper_37 if p in X_excess.columns]
print(f"overlap with paper's 37 anomalies: {len(overlap)}/37")
print(f"matched: {overlap}")

print(f"\nX matrix: {X_excess.shape}")
print(f"  mean monthly ret: {X_excess.values.mean()*100:.3f}%")
print(f"  mean monthly std: {X_excess.std().mean()*100:.3f}%")
print(f"  any NaN:          {X_excess.isnull().any().any()}")


# %%

#___ save ___

# convert period index back to timestamp for csv compatibility
X_save = X_excess.copy()
X_save.index = X_excess.index.to_timestamp()
X_save.to_csv("X_openAP.csv")
print(f"saved X_openAP.csv  --  {X_excess.shape[0]} months x {X_excess.shape[1]} anomalies")
print("next: rppca_openAP.py")

loading PredictorLSretWide.csv...
  shape: (1188, 213)
  first few cols: ['date', 'AM', 'AOP', 'AbnormalAccruals', 'Accruals']
  sample dates: ['1926-01-30', '1926-02-27', '1926-03-31']
date range in file: 1926-01 to 2024-12
avg absolute value mid-sample: 2.0500
  looks like percentages -- dividing by 100
after date filter: (726, 212)

coverage summary:
  total months:          726
  total anomalies:       212
  full coverage (100%):  132
  90%+ coverage:         145
  75%+ coverage:         178

anomalies by start year:
  1963: ██████████████████████████████████████████████████ (135)
  1964: █████ (5)
  1965: ███ (3)
  1966: ██ (2)
  1970: ███ (3)
  1971: ███ (3)
  1972: █████████ (9)
  1973: ██ (2)
  1974: █ (1)
  1975: █ (1)
  1976: ██████████ (10)
  1977: █████ (5)
  1979: █ (1)
  1980: ███ (3)
  1981: █ (1)
  1982: ██ (2)
  1983: ██ (2)
  1984: █ (1)
  1986: ███ (3)
  1988: █ (1)
  1990: ███ (3)
  1993: █████ (5)
  1994: █ (1)
  1996: █████████ (9)
  2000: █ (1)
anomalies passing 